In [1]:
from src.model import initialize_embedding_model_from_causal_lm, LlamaForEmbeddingLM
from transformers import LlamaForCausalLM, AutoTokenizer
import torch

# Initialize the LlamaForEmbeddingLM model with the pre-trained llama weights
- Make sure there has enough memory for loading 8B model

In [2]:
# Initialize embedding model from pretrained LLaMA model
model = initialize_embedding_model_from_causal_lm(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    dim_embed_domain=1024,
    dim_adapter_hidden=2048
)

# Get the tokenizer from the original model
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct"
)

Initializing embedding model with the new config (adapter will be initialized randomly)
Loading the weights from the pretrained model


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Getting the state dict from the pretrained model
Loading the state dict into our embedding model


In [3]:
# Save the model with its adapter (which is randomly initialized at this point)
output_dir = "./initial_elm_model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('./test_initial_elm_model/tokenizer_config.json',
 './test_initial_elm_model/special_tokens_map.json',
 './test_initial_elm_model/tokenizer.json')

# Load Model and Validate

In [7]:
# Later, you can load it normally
loaded_model = LlamaForEmbeddingLM.from_pretrained(
    "./initial_elm_model", 
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [5]:
# Load the pre-trained LLaMA model
llama_model = LlamaForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
# validate if the loaded_model really uses the pre-trained weights
# Compare a few weights between the loaded model and the original llama model
# Check embedding weights
print(loaded_model.model.embed_tokens.weight[0, 0:5])
print(llama_model.model.embed_tokens.weight[0, 0:5])

# Check if they are equal
assert torch.allclose(loaded_model.model.embed_tokens.weight, llama_model.model.embed_tokens.weight)

# Check some layer weights
assert torch.allclose(
    loaded_model.model.layers[0].self_attn.q_proj.weight,
    llama_model.model.layers[0].self_attn.q_proj.weight
)

assert torch.allclose(
    loaded_model.model.layers[0].self_attn.k_proj.weight,
    llama_model.model.layers[0].self_attn.k_proj.weight
)

assert torch.allclose(
    loaded_model.model.layers[30].self_attn.q_proj.weight,
    llama_model.model.layers[30].self_attn.q_proj.weight
)

assert torch.allclose(
    loaded_model.model.layers[30].self_attn.k_proj.weight,
    llama_model.model.layers[30].self_attn.k_proj.weight
)

tensor([ 0.0011,  0.0056, -0.0034, -0.0012, -0.0036], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)
tensor([ 0.0011,  0.0056, -0.0034, -0.0012, -0.0036], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)
